## Imports

In [1]:
from src.models.U_net import UNet
from src.train import model_trainer
from src.dataset.dataset import ADE20KDataset, make_train_transform, make_val_transform
from src.utils.params import Params, integrate_global_parameters
from src.preprocessing.verify_model import verify_model_file, verify_model_memorization
from src.preprocessing.hyperparameter_estimation import estimate_hyperparameters


c:\Users\Komputer\Documents\Igor\Studia\Sem5\CV\Project3-CV-segmentation-inpainting\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from src.preprocessing.presize_images import process_split

In [3]:
parameters : Params = Params('src/config/uNet_params.json')
model = UNet(in_channels=3, base_channels=64, num_classes=parameters.get("num_classes", 151))
global_parameters = Params()
global_parameters = integrate_global_parameters(global_parameters)
dataset = ADE20KDataset(global_parameters.get("train_image_folder"),
                        global_parameters.get("train_annotation_folder"),
                        transform=make_train_transform(
                            mean = global_parameters.get("mean", (0.485, 0.456, 0.406)),
                            std = global_parameters.get("std", (0.229, 0.224, 0.225))
                            )
                        )

In [ ]:
verify_model_memorization(model = model, dataset=dataset, sample_size=8, params=parameters, epochs=2)

Training on device: cpu
Using single GPU or CPU
DataLoader settings - batch_size: 8, num_workers: 4, pin_memory: False, prefetch_factor: 2


c:\Users\Komputer\Documents\Igor\Studia\Sem5\CV\Project3-CV-segmentation-inpainting\.venv\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:178: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://github.com/mlflow/mlflow-export-import can be used.
  return FileStore(store_uri, store_uri)
2026/01/27 14:01:32 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/01/27 14:01:32 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Epoch 1/150, Train Loss: 5.2310
  Validation: {'CrossEntropyLoss': 935782.875, 'DiceLoss': 0.9741854667663574, 'FocalLoss': 233804.09375, 'mIoU': 0.002371919574216008, 'pixel_accuracy': 0.036968231201171875}
  -> Best model saved with val_loss: 935782.8750
Most recent model saved to ./models/overfitted_unet_last.pt
Epoch 2/150, Train Loss: 4.3273
  Validation: {'CrossEntropyLoss': 721354624.0, 'DiceLoss': 0.9834673404693604, 'FocalLoss': 180107904.0, 'mIoU': 0.0015528275398537517, 'pixel_accuracy': 0.016817092895507812}
Most recent model saved to ./models/overfitted_unet_last.pt
Epoch 3/150, Train Loss: 3.6870
  Validation: {'CrossEntropyLoss': 295737344.0, 'DiceLoss': 0.9769977331161499, 'FocalLoss': 73746416.0, 'mIoU': 0.0020727496594190598, 'pixel_accuracy': 0.06631278991699219}
Most recent model saved to ./models/overfitted_unet_last.pt
Epoch 4/150, Train Loss: 3.2534
  Validation: {'CrossEntropyLoss': 40745504.0, 'DiceLoss': 0.978404700756073, 'FocalLoss': 10146778.0, 'mIoU': 0.00

2026/01/27 14:24:14 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/01/27 14:24:14 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Epoch 127/150, Train Loss: 0.2327
  Validation: {'CrossEntropyLoss': 0.38364189863204956, 'DiceLoss': 0.4573456645011902, 'FocalLoss': 0.05928561091423035, 'mIoU': 0.5517615675926208, 'pixel_accuracy': 0.9099006652832031}
Early stopping triggered after 127 epochs (patience: 15)
Training complete.
Model failed to memorize the small subset of data.


False

In [5]:
# estimate_hyperparameters(model=model, model_params_path='src/config/uNet_params.json',
#                          dataset=dataset, n_trials=20, sample_size=16, epochs=5)

In [6]:
from src.predict import predict_dataset
from src.dataset.dataset import get_random_subset
import torch

model : UNet= UNet(in_channels=3, base_channels=64, num_classes=151)
model.load_state_dict(torch.load('models/best_unet.pt', map_location=torch.device('cpu')))

validation_dataset = ADE20KDataset(global_parameters.get("validation_image_folder"),
                        global_parameters.get("validation_annotation_folder"),
                        transform=make_val_transform(
                            mean = global_parameters.get("mean", (0.485, 0.456, 0.406)),
                            std = global_parameters.get("std", (0.229, 0.224, 0.225))
                            )
                        )

small_sample = get_random_subset(validation_dataset, 20)
device = torch.device('cpu')
# predict(model, small_sample, device)


In [7]:
# #https://huggingface.co/microsoft/beit-large-finetuned-ade-640-640
# from transformers import BeitFeatureExtractor, BeitForSemanticSegmentation
# # from datasets import load_dataset
# from PIL import Image

# # load ADE20k image
# # ds = load_dataset("hf-internal-testing/fixtures_ade20k", split="test")

# print(type(image))
# feature_extractor = BeitFeatureExtractor.from_pretrained('microsoft/beit-base-finetuned-ade-640-640')
# model = BeitForSemanticSegmentation.from_pretrained('microsoft/beit-base-finetuned-ade-640-640') #2.5GB ma large model i mi kompa crashował więc base dałem; base ma 900MB

# inputs = feature_extractor(images=image_np, return_tensors="pt")
# outputs = model(**inputs)
# # logits are of shape (batch_size, num_labels, height/4, width/4)
# logits = outputs.logits


In [8]:
torch.__version__

'2.10.0+cpu'

## Model Comparison

In [9]:
# Load models for comparison
import torch
from src.models.U_net import UNet
from src.models.mask2former import mask2former
from src.models.convnext import Convnext
from src.dataset.dataset import get_random_subset
from src.validation.compare_models import compare_models, compare_models_visually

# Initialize device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load UNet model (trained)
unet_model = UNet(in_channels=3, base_channels=64, num_classes=151)
unet_model.load_state_dict(torch.load('models/best_unet.pt', map_location=device))
unet_model.to(device)
unet_model.eval()

print("✓ Loaded UNet model")

# Load Mask2Former (pre-trained on ADE20K)
# Note: This will download ~1GB on first run
mask2former_model = mask2former(num_classes=150)
mask2former_model.to(device)
mask2former_model.eval()

print("✓ Loaded Mask2Former model")

# Load ConvNeXt (pre-trained on ADE20K)
# Note: This will download ~400MB on first run
convnext_model = Convnext(model_id="openmmlab/upernet-convnext-base", device=device)
convnext_model.eval()
print("✓ Loaded ConvNeXt model")

# List of models to compare
models = [unet_model, mask2former_model, convnext_model]

print(f"\n{len(models)} models loaded and ready for comparison")

Using device: cpu
✓ Loaded UNet model


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Users\Komputer\Documents\Igor\Studia\Sem5\CV\Project3-CV-segmentation-inpainting\.venv\Lib\site-packages\transformers\image_processing_base.py:417: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size', 'reduce_labels'
  image_processor = cls(**image_processor_dict)


✓ Loaded Mask2Former model
✓ Loaded ConvNeXt model

3 models loaded and ready for comparison


In [10]:
# # Add this before validation to inspect model outputs
# from src.validation.compare_models import compare_models

# # Test on just 1 sample
# test_subset = get_random_subset(validation_dataset, 1)
# dummy_data_loader = torch.utils.data.DataLoader(test_subset, batch_size=1, shuffle=False)
# # Print shape and value ranges
# for data in dummy_data_loader:
#     print(data)
#     img, mask = data['image'].to(device), data['annotation'].to(device)
#     print(f"Image shape: {img.shape}, dtype: {img.dtype}")
#     print(f"Mask shape: {mask.shape}, dtype: {mask.dtype}")
#     print(f"Unique mask values: {mask.unique()}")
    
#     # Get model output

#     with torch.no_grad():
#         output = mask2former_model.predict(img.to(device))
#     print(f"Model output type: {type(output)}")
#     print(f"Model output keys/shape: {output.keys() if isinstance(output, dict) else output.shape}")
#     print(output.unique())
#     break

In [11]:
from src.predict import predict_dataset
# Compare models on a small subset for quick validation
small_sample = get_random_subset(validation_dataset, 2)
predict_dataset(models[1], small_sample)

Using single GPU or CPU
torch.Size([2, 3, 256, 256])
torch.Size([2, 151, 256, 256])
(256, 512, 3)
(256, 512, 3)
Predictions saved to ./output/mask2former/


In [12]:
# # Diagnostic: Check mask2former output and ground truth alignment
# test_subset = get_random_subset(validation_dataset, 1)
# test_loader = torch.utils.data.DataLoader(test_subset, batch_size=1, shuffle=False)

# for data in test_loader:
#     img = data['image'].to(device)
#     gt_mask = data['annotation'].to(device)
    
#     print("=" * 60)
#     print("GROUND TRUTH")
#     print("=" * 60)
#     print(f"GT shape: {gt_mask.shape}")
#     print(f"GT dtype: {gt_mask.dtype}")
#     print(f"GT unique values: {torch.unique(gt_mask)}")
#     print(f"GT min/max: {gt_mask.min()}/{gt_mask.max()}")
    
#     print("\n" + "=" * 60)
#     print("MASK2FORMER FORWARD OUTPUT (logits)")
#     print("=" * 60)
#     with torch.no_grad():
#         logits = mask2former_model.forward(img)
#     print(f"Logits shape: {logits.shape}")
#     print(f"Logits dtype: {logits.dtype}")
#     print(f"Logits contains -inf: {torch.isinf(logits).any()}")
#     print(f"Logits channel 0 (background) stats:")
#     print(f"  Min: {logits[:, 0].min()}, Max: {logits[:, 0].max()}")
#     print(f"Logits channel 1-150 stats:")
#     print(f"  Min: {logits[:, 1:].min()}, Max: {logits[:, 1:].max()}")
    
#     print("\n" + "=" * 60)
#     print("MASK2FORMER PREDICT OUTPUT (class indices)")
#     print("=" * 60)
#     with torch.no_grad():
#         pred_mask = mask2former_model.predict(img)
#     print(f"Prediction shape: {pred_mask.shape}")
#     print(f"Prediction dtype: {pred_mask.dtype}")
#     print(f"Prediction unique values: {torch.unique(pred_mask)}")
#     print(f"Prediction min/max: {pred_mask.min()}/{pred_mask.max()}")
    
#     print("\n" + "=" * 60)
#     print("ALIGNMENT CHECK")
#     print("=" * 60)
#     print(f"Does prediction ever predict class 0 (background)? {(pred_mask == 0).any()}")
#     print(f"GT has background pixels (class 0)? {(gt_mask == 0).any()}")
#     print(f"Prediction classes in GT: {torch.unique(pred_mask[torch.isin(pred_mask, gt_mask)])}")
    
#     break

In [13]:
# Quantitative comparison on validation subset
from src.utils.params import Params

# Create a small validation subset for quick comparison
val_subset = get_random_subset(validation_dataset, 100)  # Use 100 samples

# Set up parameters for validation
comparison_params = Params()
comparison_params.set("device", device)
comparison_params.set("batch_size", 2)
comparison_params.set("num_workers", 0)  # Windows compatibility
comparison_params.set("num_classes", 151)
comparison_params.set("ignore_index", 0)
comparison_params.set("validation_loss_functions", {
    "CrossEntropyLoss": "CrossEntropyLoss",
    "DiceLoss": "DiceLoss"
})
comparison_params.set("validation_loss_params", {"ignore_index": 0})

print("Starting quantitative comparison on 100 validation samples...")
print("This may take a few minutes...\n")

# Run comparison
results = compare_models(
    models=models[:1],
    dataset=val_subset,
    params=comparison_params,
    output_directory="./output/model_comparison/"
)

# Display results
print("\n" + "="*60)
print("QUANTITATIVE COMPARISON RESULTS")
print("="*60)
for res in results:
    print(f"\n{res['model_name']}:")
    for metric, value in res['results'].items():
        print(f"  {metric:20s}: {value:.4f}")

print("\n✓ Results saved to: ./output/model_comparison/comparison_results.csv")

Starting quantitative comparison on 100 validation samples...
This may take a few minutes...

Validating model: UNet
Using single GPU or CPU

QUANTITATIVE COMPARISON RESULTS

UNet:
  CrossEntropyLoss    : 1.9799
  DiceLoss            : 0.6946
  mIoU                : 0.1325
  pixel_accuracy      : 0.5197

✓ Results saved to: ./output/model_comparison/comparison_results.csv


In [14]:
# Visual comparison on a few samples
print("Creating visual comparison...")

# Set parameters for visual comparison
visual_params = Params()
visual_params.set("input_height", 512)
visual_params.set("input_width", 512)

# Compare visually on 5 random samples
compare_models_visually(
    models=models,
    dataset=validation_dataset,
    params=visual_params,
    output_directory="./output/visual_comparison/",
    sample_number=10
)

print("✓ Visual comparisons saved to: ./output/visual_comparison/")
print("\nCheck the output folder to see side-by-side model predictions!")

Creating visual comparison...
torch.Size([1, 256, 256])
torch.Size([1, 3, 256, 256])
torch.Size([1, 151, 256, 256])
torch.Size([1, 256, 256])
torch.Size([1, 256, 256])
Combined image size will be calculated for sample 0
[0.7  0.21 0.21]
(256, 256, 3)
torch.Size([1, 256, 256])
torch.Size([1, 3, 256, 256])
torch.Size([1, 151, 256, 256])
torch.Size([1, 256, 256])
torch.Size([1, 256, 256])
Combined image size will be calculated for sample 1
[0.7        0.09759124 0.07      ]
(256, 256, 3)
torch.Size([1, 256, 256])
torch.Size([1, 3, 256, 256])
torch.Size([1, 151, 256, 256])
torch.Size([1, 256, 256])
torch.Size([1, 256, 256])
Combined image size will be calculated for sample 2
[0.7  0.21 0.21]
(256, 256, 3)
torch.Size([1, 256, 256])
torch.Size([1, 3, 256, 256])
torch.Size([1, 151, 256, 256])
torch.Size([1, 256, 256])
torch.Size([1, 256, 256])
Combined image size will be calculated for sample 3
[0.9  0.54 0.27]
(256, 256, 3)
torch.Size([1, 256, 256])
torch.Size([1, 3, 256, 256])
torch.Size([1

In [15]:
# Display one visual comparison in the notebook
from IPython.display import Image as IPImage, display

# Show the first comparison image
comparison_path = "./output/visual_comparison/sample_0/comparison.png"
try:
    display(IPImage(filename=comparison_path))
    print(f"Showing: {comparison_path}")
except FileNotFoundError:
    print(f"Run the previous cell first to generate comparison images!")

Run the previous cell first to generate comparison images!
